# 🎯 [K09_M02] 부대 안내 자료 정리하기
### 학생 실습 워크북 & 프롬프트 설계 가이드 (Google Colab / Jupyter)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/K09/k09_m02_student.ipynb)

---

## 📌 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 / 과제** | 병영생활·상담 (Level 2) / K09 부대 정보 검색 시스템 |
| **미션** | 미션 02. 부대 안내 자료 정리하기 (소요시간: 약 1.5시간) |
| **실습 유형** | 생성형 AI 프롬프트 엔지니어링 및 팩트체크 검증 |

> **⚠️ [교육용 가상 자료 원칙]**
> 본 실습의 모든 자료는 안전 규정에 따라 **교육용 가상 부대 자료(가상 한빛대대)**로 구성되어 있습니다.
> 실제 군사 기밀이나 부대 세부 인적·작전 정보는 외부 AI 도구에 절대 입력하지 않습니다.



In [ ]:
# ============================================================================
# [환경 설정 및 헬퍼 함수] 마크다운 시각화 렌더러 및 가상 데이터셋 로더
# ============================================================================
from IPython.display import display, Markdown
import json
import os

def load_dataset(mission_num):
    """미션 번호(1~10)에 해당하는 가상 데이터셋 JSON을 로드하여 반환합니다."""
    filename = f"k09_m{mission_num:02d}_dataset.json"
    for base in [".", "uploads/K09", "../uploads/K09"]:
        filepath = os.path.join(base, filename)
        if os.path.exists(filepath):
            with open(filepath, "r", encoding="utf-8") as f:
                return json.load(f)
    print(f"⚠️ 데이터셋 파일을 찾을 수 없습니다: {filename}")
    return None

def show_dataset_summary(mission_num):
    """해당 미션의 데이터셋 주요 항목을 요약 출력합니다."""
    data = load_dataset(mission_num)
    if data:
        display(Markdown(f"**[데이터셋 로드 성공]** `{data.get('title', '')}` ({data.get('mission', '')})"))
        keys = [k for k in data.keys() if k not in ["project", "mission", "title"]]
        print(f"포함 항목: {', '.join(keys)}")

def show_result(mission_title, prompt, result):
    """학습자가 입력한 프롬프트와 AI 답변 및 검증 결과를 시각적으로 렌더링하는 헬퍼 함수"""
    p_text = prompt.strip() if prompt.strip() else "(작성된 프롬프트가 없습니다)"
    r_text = result.strip() if result.strip() else "(기록된 결과가 없습니다)"
    
    md_content = f"""
---
### 📌 [{mission_title}] 실습 기록 리포트

#### 💬 입력 프롬프트
```text
{p_text}
```

#### 📋 AI 생성 결과 및 팩트체크 검증 내역
{r_text}
---
"""
    display(Markdown(md_content))

print("✅ 워크북 시각화 헬퍼 함수 및 가상 데이터셋 로더(k09_m01~m10) 준비 완료!")



---

## 🎯 미션 02. 부대 안내 자료 정리하기
**소요시간**: 약 1.5시간

![미션 2 부대 안내 줄글 데이터의 5대 표준 표 구조화 다이어그램](/uploads/common/K09_m02_data_structuring.png)

### 📋 미션 개요 및 배경
생성형 AI가 거짓 정보를 생성하지 않도록 신뢰할 수 있는 부대 생활 참조 데이터를 체계적으로 구조화합니다. 가상 한빛대대 시설 안내 12건을 시설명·위치·운영 시간·연락처·제공 업무 항목으로 표준화된 표로 정리하고, 소독 시간이나 야간 당직 등 예외 운영 조건을 명시합니다. AI가 정밀하게 참조할 수 있는 비정형 텍스트의 표 구조화(데이터 정제) 역량을 체득합니다.

### 🎯 미션 학습목표
가상 한빛대대 시설 안내 12건을 시설명·위치·운영 시간·연락처·제공 업무 항목의 표로 정리하고, 소독 시간 및 야간 당직 등 운영 예외 조건을 명확히 표시합니다. **이를 통해 비정형 텍스트 안내문을 AI가 정확히 검색하고 파싱할 수 있는 표준 데이터 포맷으로 전처리·구조화하는 데이터 정제 역량을 익힙니다.**

### 📑 실습 가이드 및 세부 단계 (프롬프트 & 검증 포인트)
#### 🔹 Step 1. 시설 12건을 자료표 항목으로 정리한다.
#### 이 단계의 역할
가상 한빛대대의 줄글형 시설 안내문 12건을 AI가 정확히 탐색할 수 있는 표준 표(Table) 구조로 정제합니다.

#### 해야 할 일

<li>가상 한빛대대 시설 안내문 12건 원문을 꼼꼼히 확인합니다.

</li><li>[시설명, 위치, 운영 시간, 문의처, 제공 업무]의 5대 필수 컬럼으로 구성된 표 서식을 준비합니다.

</li><li>AI에게 줄글 텍스트를 마크다운 표로 변환하도록 요청하고, 변환된 표의 필드를 원문과 대조합니다.

</li>

#### 💡 AI 프롬프트 질의 예시
<blockquote>**[시설 안내 데이터 표 구조화 질의]:** "너는 데이터 구조화 전문가야. 아래 부대 시설 안내 줄글 12건을 [시설명, 위치, 운영 시간, 문의처, 제공 업무] 컬럼을 가진 마크다운 표(Table) 형식으로 일목요연하게 변환해 줘. 빠진 내용이 없어야 해.
[안내문 12건]: {가상 시설 안내 텍스트}"

</blockquote>
#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **12건 전수 포함:** 의무실, 사이버지식정보방, 체력단련실, PX 등 12개 시설이 누락 없이 표에 기재되었는가?
- **컬럼 규격화:** 5대 필수 항목이 공통 서식으로 빈틈없이 채워졌는가?

#### 핵심 확인 지표
<blockquote>12건 부대 시설 표준 자료표 초안

</blockquote>
#### 학습자가 제출해야 하는 내용

<li>부대 생활 안내 자료표(가상) 초안

</li>

#### 🔹 Step 2. 운영 시간 예외(소독·야간 당직)를 표시한다.
#### 이 단계의 역할
정규 운영 시간 외에 점심 소독 시간(Break time), 주말/공휴일 단축, 야간 당직 진료 등 특이 조건을 별도 식별하여 표시합니다.

#### 해야 할 일

<li>12개 시설의 원문에서 '단, ~는 제외', '소독 시간', '당직 접수' 등 예외 조항을 색출합니다.

</li><li>표의 '운영 시간' 컬럼에 정규 시간과 예외 조건을 괄호 또는 별도 표기로 명확히 구분하여 기재합니다.

</li><li>AI에게 조건부 예외 운영이 존재하는 시설 목록을 추출하도록 질의하여 누락을 크로스체크합니다.

</li>

#### 💡 AI 프롬프트 질의 예시
<blockquote>**[운영 예외 조건 추출 질의]:** "정리된 시설 안내표에서 점심시간 소독, 정기 방역, 야간 당직, 주말 단축 등 '정규 시간 외 예외 조건'이 존재하는 시설들만 따로 추출하여 예외 내용과 이용 시 주의사항을 정리해 줘."

</blockquote>
#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **예외 조건 정확도:** 의무실 소독 시간(12:00~13:30)이나 당직 접수 조건 등이 원문과 일치하는가?
- **시각적 가독성:** 정규 운영 시간과 예외 조건이 혼동되지 않게 명확히 구분 표기되었는가?

#### 핵심 확인 지표
<blockquote>예외 운영 조건이 보강된 부대 생활 안내 자료표

</blockquote>
#### 학습자가 제출해야 하는 내용

<li>부대 생활 안내 자료표(가상) 완성본

</li>

#### 🔹 Step 3. 심화 도전 과제 및 최적화
#### 이 단계의 역할
안내 자료표에 '자주 묻는 질문(FAQ)' 열을 추가하여 AI가 장병들의 질문 의도와 답변을 즉각 매핑할 수 있도록 데이터셋을 확장합니다.

#### 해야 할 일
자료표에 각 시설별 '자주 묻는 질문' 칸을 추가하여 데이터를 보강해 본다.

#### 💡 AI 프롬프트 질의 예시
<blockquote>**[시설별 FAQ 확장 질의]:** "아래 12개 시설 안내표에 각 시설마다 장병들이 실제로 물어볼 만한 대표 질문 2개씩을 추가하여 '자주 묻는 질문(FAQ)' 컬럼을 확장한 마크다운 표로 만들어 줘.
[자료표]: {완성된 자료표}"

</blockquote>
#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **구어체 질문 반영:** 실제 장병들이 사용하는 자연스러운 질문 형태가 반영되었는가?
- **데이터 일관성:** 확장된 질문들이 기존 시설 정보 범위 내에서 답변 가능한가?

#### 핵심 확인 지표
<blockquote>FAQ 열이 확장된 고도화 부대 안내 자료표

</blockquote>
#### 학습자가 제출해야 하는 내용

<li>심화 확장 부대 생활 안내 자료표

</li>

### 📝 실습 서식 및 모범 작성 양식 (참고 템플릿)
# [K09_M02] 부대 안내 자료 정리하기 실습 가이드 및 작성 서식

## 1. 부대 생활 안내 자료표 (가상 한빛대대 표준 데이터셋)
- 가상 한빛대대 시설 12건의 줄글 원문을 바탕으로 AI가 검색하기 용이한 5대 표준 컬럼 표로 구조화합니다.

| 연번 | 시설명 | 위치 | 정규 운영 시간 | 예외 운영 조건 (소독/휴무/단서) | 문의 연락처 | 주요 제공 업무 |
| :---: | :--- | :--- | :--- | :--- | :--- | :--- |
| 1 | 대대 의무실 | 본관 1층 서편 | 평일 09:00~17:30 | 12:00~13:30 소독(진료 불가), 야간/휴일 당직 접수 | 내선 103 | 일반 진료, 약 조제, 물리치료 |
| 2 | 사이버지식정보방 | 복지회관 2층 201호 | 평일 17:30~21:00, 주말 09:00~21:00 | 매주 화요일 20:30~21:00 정기 점검 | 내선 205 | 인터넷 PC, 원격강좌, 자격증 접수 |
| 3 | 체력단련실 | 본관 지하 1층 | 평일 06:30~21:00, 주말 08:00~21:00 | 평일 일과시간 개인 이용 불가, 매주 수 14~16시 방역 | 내선 104 | 웨이트 트레이닝, 유산소, 샤워장 |
| 4 | 복지매장(PX) | 복지회관 1층 | 평일 12:00~19:30, 토 10:00~18:00 | 매주 일요일 정기 휴무, 평일 14~15시 재고조사 | 내선 201 | 생필품, 식음료, 택배 접수 |
| 5 | 병영도서관 | 복지회관 2층 204호 | 화~일 13:00~21:00 | 매주 월요일 정기 휴관, 1인 3권 14일 대출 | 내선 208 | 도서 대출 및 열람, 신문 구독 |
| 6 | 병영 세탁실 | 각 생활관 층별 중앙 | 매일 06:00~21:30 | 점호 준비(21:30 이후) 작동 금지, 분대별 요일제 | 각 중대 행정반 | 세탁기, 건조기, 다림질 |
| 7 | 부대 이발실 | 복지회관 지하 1층 | 화·목·토 13:30~17:30 | 사전 중대별 예약제 (선착순 방문 불가) | 내선 203 | 군 표준 두발 정리 |
| 8 | 영내 종교관 | 연병장 우측 종교시설동 | 수 19:00~20:30, 일 09:30~12:00 | 평일 개인 방문 시 군종병 사전 승인 필수 | 내선 109 | 주말 정기 예배/법회/미사, 종교 상담 |
| 9 | 우편물보관소 | 본관 1층 당직사령실 옆 | 평일 17:30~18:30 (수령 시간) | 사제 택배는 위병소 보안검색 후 익일 수령 | 내선 101 | 등기, 소포, 택배 수령 및 발송 |
| 10 | 병영생활상담실 | 본관 2층 동편 | 평일 09:00~18:00 | 100% 비밀 보장, 24시간 긴급 핫라인 운영 | 내선 119 | 1:1 고충 상담, 심리검사 |
| 11 | 가족 면회실 | 위병소 옆 단독 건물 | 토·일·공휴일 09:00~17:00 | 평일 면회 불가, 최소 3일 전 행정반 신청 | 내선 102 | 면회, 배달음식 취식, 휴게 공간 |
| 12 | 한빛 병영식당 | 본관 뒤편 급양시설동 | 조식 07~08, 중식 12~13, 석식 17:30~18:30 | 매주 토요일 브런치데이(09:00~10:30) 운영 | 내선 105 | 정규 급식, 식단표 공시, 브런치 |

### 📦 필수 학습자료 및 실습 팁
**미션 수행 필수 학습자료**

- **가상 한빛대대 시설 안내문 12건 원문 (MD):** 의무실, PX, 체력단련실, 사지방 등 12개 부대 핵심 시설 줄글 안내문
- **표준 부대 생활 안내 자료표 서식 (Markdown Table):** 시설명, 위치, 정규 운영 시간, 연락처, 업무 5대 표준 컬럼 템플릿
- **[학습 가이드] 비정형 텍스트의 표(Table) 구조화 및 데이터 정제 기법:** 생성형 AI가 오류 없이 파싱할 수 있도록 예외 조항(소독, 당직)을 명시하는 데이터 전처리 가이드

**실습 팁:** '점심시간 소독(12:00~13:30)'이나 '야간 당직 진료' 같은 예외 조건을 누락하면 AI가 진료 불가 시간에 장병을 헛걸음하게 만들 수 있으므로 반드시 표기하세요.

### 🔍 평가 기준 (PASS 조건)
#### 미션 판정
**평가 구분:** 필수

**핵심 평가 기준:** [필수] 정보 구조화 — AI 검색에 최적화된 5대 표준 컬럼으로 안내 자료를 정리하고 운영 예외 조건을 누락 없이 표시했는가? (PASS / FAIL)

#### 미션 PASS 조건

<li>12개 부대 시설 표준 안내 자료표 완성도 충족

</li><li>점심 소독 시간, 야간 당직, 요일제 등 운영 예외 조건 100% 반영

</li><li>가상 원문 데이터와 수치(시간, 위치, 내선번호) 1:1 일치

</li>

#### 채점자 판정 메모
<blockquote>의무실 점심 소독 시간(12:00~13:30)과 체력단련실 방역 시간 등 장병 헛걸음을 유발할 수 있는 예외 조건이 빠짐없이 적혔는지 육안 대조한다.

</blockquote>





In [ ]:
# [미션 02. 부대 안내 자료 정리하기] 학생 실습 프롬프트 및 결과 기록란
mission_02_prompt = """
# [TODO: 미션 02] 줄글 형태의 시설 안내문 12건을 5대 표준 컬럼 표로 구조화하는 프롬프트를 작성하세요.
# 힌트: load_dataset(2)의 facilities 항목을 확인하고, 예외 운영 조건(소독, 점검, 야간)이 누락되지 않도록 지시합니다.
# (여기에 작성한 프롬프트를 입력하세요)
"""

mission_02_result = """
### 📋 [미션 02 실습 결과]
## 1. 부대 생활 안내 자료표 (12개 시설 표준 데이터셋)
| 연번 | 시설명 | 위치 | 정규 운영 시간 | 예외 운영 조건 | 문의 연락처 | 주요 제공 업무 |
| :---: | :--- | :--- | :--- | :--- | :--- | :--- |
| 1 | 대대 의무실 | | | | | |
| 2 | 사지방 | | | | | |
| 3 | 체력단련실 | | | | | |
| ... | | | | | | |
"""

# 렌더러 실행 (결과를 시각적으로 확인합니다)
show_result("미션 02. 부대 안내 자료 정리하기", mission_02_prompt, mission_02_result)

